In [ ]:
!pip install --upgrade -q "google-meridian[and-cuda]"

import numpy as np
import pandas as pd
import altair as alt
import tensorflow as tf
import tensorflow_probability as tfp
from IPython.display import display, HTML

from meridian import constants
from meridian.data import load
from meridian.model import model
from meridian.model import spec
from meridian.model import prior_distribution
from meridian.analysis import analyzer
from meridian.analysis import visualizer
from meridian.analysis import optimizer
from meridian.analysis import summarizer

def show(chart_or_obj, title=None):
    if title:
        display(HTML(f"<h3 style='font-family:sans-serif'>{title}</h3>"))
    display(chart_or_obj)

print("TensorFlow:", tf.__version__)
gpus = tf.config.experimental.list_physical_devices("GPU")
print("GPUs detected:", gpus if gpus else "NONE — sampling will be slow on CPU!")

CSV_URL = (
    "https://raw.githubusercontent.com/google/meridian/refs/heads/main/"
    "meridian/data/simulated_data/csv/geo_all_channels.csv"
)
df = pd.read_csv(CSV_URL)

print("\nShape:", df.shape)
print("Geos:", df["geo"].nunique(), "| Weeks:", df["time"].nunique())
print("Date range:", df["time"].min(), "->", df["time"].max())
display(df.head())

spend_cols = [c for c in df.columns if c.endswith("_spend")]
spend_share = df[spend_cols].sum().rename("total_spend").reset_index()
spend_share["share_%"] = 100 * spend_share["total_spend"] / spend_share["total_spend"].sum()
display(spend_share)

kpi_by_week = df.groupby("time")["conversions"].sum().reset_index()
show(
    alt.Chart(kpi_by_week).mark_line().encode(
        x=alt.X("time:T", title="Week"),
        y=alt.Y("conversions:Q", title="Total conversions (all geos)"),
    ).properties(width=700, height=250),
    "National KPI over time",
)

In [ ]:
coord_to_columns = load.CoordToColumns(
    time="time",
    geo="geo",
    controls=["competitor_sales_control", "sentiment_score_control"],
    population="population",
    kpi="conversions",
    revenue_per_kpi="revenue_per_conversion",
    media=[
        "Channel0_impression",
        "Channel1_impression",
        "Channel2_impression",
        "Channel3_impression",
        "Channel4_impression",
    ],
    media_spend=[
        "Channel0_spend",
        "Channel1_spend",
        "Channel2_spend",
        "Channel3_spend",
        "Channel4_spend",
    ],
    organic_media=["Organic_channel0_impression"],
    non_media_treatments=["Promo"],
)

media_to_channel = {f"Channel{i}_impression": f"Channel_{i}" for i in range(5)}
media_spend_to_channel = {f"Channel{i}_spend": f"Channel_{i}" for i in range(5)}

loader = load.CsvDataLoader(
    csv_path=CSV_URL,
    kpi_type="non_revenue",
    coord_to_columns=coord_to_columns,
    media_to_channel=media_to_channel,
    media_spend_to_channel=media_spend_to_channel,
)
data = loader.load()
print("\nInputData loaded. Media tensor shape (geo, time, channel):", data.media.shape)

roi_mu = 0.2
roi_sigma = 0.9
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M)
)

model_spec = spec.ModelSpec(prior=prior)

mmm = model.Meridian(input_data=data, model_spec=model_spec)

In [ ]:
mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=7,
    n_adapt=500,
    n_burnin=500,
    n_keep=1000,
    seed=1,
)
print("Sampling complete.")

model_diagnostics = visualizer.ModelDiagnostics(mmm)
show(model_diagnostics.plot_rhat_boxplot(), "R-hat convergence check (want < 1.05)")

show(
    model_diagnostics.plot_prior_and_posterior_distribution(),
    "Prior vs. posterior (ROI parameters)",
)

model_fit = visualizer.ModelFit(mmm)
show(model_fit.plot_model_fit(), "Model fit: expected vs. actual outcome")

display(model_diagnostics.predictive_accuracy_table())

media_summary = visualizer.MediaSummary(mmm)

display(media_summary.summary_table())

show(media_summary.plot_channel_contribution_area_chart(),
     "Outcome decomposition over time (baseline + channels)")
show(media_summary.plot_contribution_pie_chart(),
     "Share of outcome: baseline vs. media")
show(media_summary.plot_spend_vs_contribution(),
     "Spend share vs. contribution share (spot over/under-investment)")
show(media_summary.plot_roi_bar_chart(),
     "ROI by channel (with credible intervals)")
show(media_summary.plot_roi_vs_effectiveness(),
     "ROI vs. effectiveness (bubble = spend)")
show(media_summary.plot_roi_vs_mroi(),
     "ROI vs. marginal ROI — mROI drives optimization, not average ROI")

In [ ]:
media_effects = visualizer.MediaEffects(mmm)

show(media_effects.plot_response_curves(),
     "Response curves (incremental outcome vs. spend)")

show(media_effects.plot_adstock_decay(),
     "Adstock decay by channel")

show(media_effects.plot_hill_curves(),
     "Hill saturation curves by channel")

analysis = analyzer.Analyzer(mmm)

roi_draws = analysis.roi()
roi_np = np.asarray(roi_draws)
channels = list(data.media_channel.values)
roi_table = pd.DataFrame({
    "channel": channels,
    "roi_mean": roi_np.mean(axis=(0, 1)),
    "roi_p05": np.quantile(roi_np, 0.05, axis=(0, 1)),
    "roi_p95": np.quantile(roi_np, 0.95, axis=(0, 1)),
})
print("\nPosterior ROI summary (custom, from raw draws):")
display(roi_table)

p_better = (roi_np[..., 1] > roi_np[..., 0]).mean()
print(f"P(ROI Channel_1 > ROI Channel_0) = {p_better:.1%}")

summary_metrics = analysis.summary_metrics()
print("\nsummary_metrics() xarray variables:", list(summary_metrics.data_vars))

inc_outcome = np.asarray(analysis.incremental_outcome())
print("Incremental outcome draws shape (chains, draws, channels):", inc_outcome.shape)

In [2]:
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

show(optimization_results.plot_budget_allocation(),
     "Optimized budget allocation")
show(optimization_results.plot_spend_delta(),
     "Recommended spend change per channel")
show(optimization_results.plot_incremental_outcome_delta(),
     "Incremental outcome gained by reallocating")
show(optimization_results.plot_response_curves(),
     "Response curves with current vs. optimal spend points")

flexible_results = budget_optimizer.optimize(
    fixed_budget=False,
    target_roi=1.5,
)
show(flexible_results.plot_budget_allocation(),
     "Flexible-budget allocation at target ROI = 1.5")

mmm_summarizer = summarizer.Summarizer(mmm)
mmm_summarizer.output_model_results_summary(
    "model_results_summary.html", "/content", "2021-01-25", "2024-01-15"
)
optimization_results.output_optimization_summary(
    "budget_optimization_summary.html", "/content"
)
print("Reports written to /content/model_results_summary.html "
      "and /content/budget_optimization_summary.html")

save_path = "/content/saved_mmm.pkl"
model.save_mmm(mmm, save_path)
mmm_reloaded = model.load_mmm(save_path)
print("Model saved and reloaded from", save_path)

roi_reloaded = np.asarray(analyzer.Analyzer(mmm_reloaded).roi()).mean(axis=(0, 1))
print("Reloaded ROI means:", np.round(roi_reloaded, 3))

print("\n" + "=" * 70)
print("TUTORIAL COMPLETE ✔")
print("Next steps with YOUR data:")
print("  1. Replace CSV_URL and CoordToColumns with your columns.")
print("  2. Calibrate per-channel ROI priors with experiment results.")
print("  3. Check R-hat < 1.05 before trusting any output.")
print("  4. Use holdout_id in ModelSpec for out-of-sample validation.")
print("=" * 70)

TensorFlow: 2.20.0
GPUs detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Shape: (6240, 20)
Geos: 40 | Weeks: 156
Date range: 2021-01-25 -> 2024-01-15


,Unnamed: 0,geo,time,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,Channel4_impression,competitor_sales_control,sentiment_score_control,Channel0_spend,Channel1_spend,Channel2_spend,Channel3_spend,Channel4_spend,Organic_channel0_impression,Promo,conversions,revenue_per_conversion,population
0,0,Geo0,2021-01-25,280668,0,0,470611,108010,-1.338765,0.115581,2058.0608,0.00000,0.00000,3667.3965,841.6044,97320,0.000000,1954576.8,0.020055,136670.94
1,1,Geo0,2021-02-01,366206,182108,19825,527702,252506,0.893645,0.944224,2685.2874,1755.74540,147.31808,4112.2974,1967.5044,201441,0.000000,2064249.6,0.020103,136670.94
2,2,Geo0,2021-02-08,197565,230170,0,393618,184061,-0.284549,-1.290579,1448.6895,2219.12230,0.00000,3067.4023,1434.1870,0,0.683819,2086382.8,0.019929,136670.94
3,3,Geo0,2021-02-15,140990,66643,0,326034,201729,-1.034740,-1.084514,1033.8406,642.52057,0.00000,2540.7310,1571.8545,0,1.289055,2826431.5,0.019987,136670.94
4,4,Geo0,2021-02-22,399116,164991,0,381982,153973,-0.319276,-0.017503,2926.6072,1590.71640,0.00000,2976.7249,1199.7440,0,0.227739,3551929.2,0.020000,136670.94


,index,total_spend,share_%
0,Channel0_spend,4.050171e+07,18.452444
1,Channel1_spend,3.132030e+07,14.269422
2,Channel2_spend,1.207935e+07,5.503311
3,Channel3_spend,8.786036e+07,40.028884
4,Channel4_spend,4.773068e+07,21.745939


alt.Chart(...)


InputData loaded. Media tensor shape (geo, time, channel): (40, 156, 5)


...

Reports written to /content/model_results_summary.html and /content/budget_optimization_summary.html


/tmp/ipykernel_518/1338847095.py:340: DeprecationWarning: save_mmm is deprecated and will be removed in a future release. Please use `schema.serde.meridian_serde.save_meridian` instead. See https://developers.google.com/meridian/docs/user-guide/saving-model-object for details.
  model.save_mmm(mmm, save_path)
/tmp/ipykernel_518/1338847095.py:341: DeprecationWarning: load_mmm is deprecated and will be removed in a future release. Please use `meridian.schema.serde.meridian_serde.load_meridian` instead. See https://developers.google.com/meridian/docs/user-guide/saving-model-object for details.
  mmm_reloaded = model.load_mmm(save_path)


Model saved and reloaded from /content/saved_mmm.pkl


/tmp/ipykernel_518/1338847095.py:345: DeprecationWarning: The `meridian` argument is deprecated and will be removed in a future version. Use `model_context` instead.
  roi_reloaded = np.asarray(analyzer.Analyzer(mmm_reloaded).roi()).mean(axis=(0, 1))


Reloaded ROI means: [1.523 0.926 1.425 0.838 1.651]

TUTORIAL COMPLETE ✔
Next steps with YOUR data:
  1. Replace CSV_URL and CoordToColumns with your columns.
  2. Calibrate per-channel ROI priors with experiment results.
  3. Check R-hat < 1.05 before trusting any output.
  4. Use holdout_id in ModelSpec for out-of-sample validation.
